# FruitBlend24: Workflow readiness
ตรวจข้อมูลสำหรับวาง workflow และแบ่งงาน agents โดยไม่แก้ workbook ต้นฉบับ

ใช้ Python kernel ที่มี pandas และ openpyxl แล้วเลือก Run All รันได้จากโฟลเดอร์โปรเจกต์หรือ notebooks หรือรัน planning/run_readiness_notebook.py โดยไม่ต้องติดตั้ง Jupyter
ผลนี้เป็น structural audit และ candidate preparation ยังไม่ใช่บทวิเคราะห์ธุรกิจครบ 6 tasks หรือการเปิด agents อัตโนมัติ

Logic อยู่ใน planning/profile_inputs.py; notebook แสดงเฉพาะสรุป และใช้ cache เมื่อ source และโค้ดไม่เปลี่ยน

In [1]:
from pathlib import Path
import hashlib, json, subprocess, sys
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'question/test.md').is_file() and (p / 'planning/profile_inputs.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the dataintern project.')
source = ROOT / 'data/raw/FruitBlend24_Intern_Case_Data.xlsx'
script = ROOT / 'planning/profile_inputs.py'
report_path = ROOT / 'planning/data_readiness.json'
sha256 = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
before_hash = sha256(source)
FORCE_REFRESH = False
cached = json.loads(report_path.read_text(encoding='utf-8')) if report_path.exists() else {}
cache_ok = (cached.get('source_sha256') == before_hash and cached.get('profile_script_sha256') == sha256(script))
if FORCE_REFRESH or not cache_ok:
    result = subprocess.run([sys.executable, str(script), '--quiet'], cwd=ROOT, capture_output=True, text=True, encoding='utf-8')
    if result.returncode:
        raise RuntimeError(result.stderr[-4000:] or result.stdout[-4000:])
report = json.loads(report_path.read_text(encoding='utf-8'))
assert sha256(source) == before_hash, 'Source workbook changed during audit'
assert report['source_sha256'] == before_hash
assert report['profile_script_sha256'] == sha256(script)
print('Audit refreshed.' if FORCE_REFRESH or not cache_ok else 'Reused audit: source and code unchanged.')
print('Source workbook unchanged. Full results: planning/data_readiness.json')

Reused audit: source and code unchanged.
Source workbook unchanged. Full results: planning/data_readiness.json


In [2]:
raw, prep, waste = report['raw_orders'], report['candidate_preparation'], report['waste']
summary = pd.DataFrame([
    ('Raw order rows', raw['rows']),
    ('Raw product names', raw['distinct_raw_names']),
    ('Exact duplicate excess rows', raw['exact_duplicate_excess_rows']),
    ('Non-menu rows after deduplication', prep['unmapped_rows_after_dedup']),
    ('Retained candidate menu rows', prep['retained_menu_rows']),
    ('Sales rows without matching weekly fruit cost', prep['missing_exact_week_fruit_cost_rows']),
    ('Waste rows without estimated cost', waste['missing_estimated_waste_cost_rows']),
    ('Wasted cups with missing estimated cost', waste['units_with_missing_estimated_waste_cost']),
], columns=['Check', 'Value'])
print(f"History: {raw['period_start']} to {raw['period_end']}; outlook: Sep-Nov 2026")
print(summary.to_string(index=False))
assert raw['rows'] == raw['exact_duplicate_excess_rows'] + prep['unmapped_rows_after_dedup'] + prep['retained_menu_rows']
print('Candidate row-count bridge reconciles. Unit/revenue bridges are required before clean-data release.')

History: 2025-09-01 to 2026-08-31; outlook: Sep-Nov 2026
                                        Check  Value
                               Raw order rows 124397
                            Raw product names     31
                  Exact duplicate excess rows    372
            Non-menu rows after deduplication    370
                 Retained candidate menu rows 123655
Sales rows without matching weekly fruit cost     75
            Waste rows without estimated cost      4
      Wasted cups with missing estimated cost     96
Candidate row-count bridge reconciles. Unit/revenue bridges are required before clean-data release.


In [3]:
roles = [
    ('A0', 'Orchestrator', 'Scope, assumptions, dispatch, final completeness'),
    ('A1', 'Data Auditor', 'Data inventory and readiness issues'),
    ('A2', 'Data Preparation & Metrics', 'Versioned clean data and shared metric contract'),
    ('A3', 'Commercial Analyst', 'Demand, pricing, promotions'),
    ('A4', 'Finance & Portfolio', 'Historical/projected P&L and budget comparison'),
    ('A5', 'Inventory & Forecast', 'Backtest, demand forecast, cup-based inventory policy'),
    ('A6', 'Independent QA', 'Reconciliation, evidence, forecast and presentation checks'),
    ('A7', 'Decision & Presentation', 'Recommendations, memo, charts and presentation'),
]
print(pd.DataFrame(roles, columns=['ID', 'Role', 'Responsibility']).to_string(index=False))
print('Design only: this notebook does not launch agents.')

ID                       Role                                             Responsibility
A0               Orchestrator           Scope, assumptions, dispatch, final completeness
A1               Data Auditor                        Data inventory and readiness issues
A2 Data Preparation & Metrics            Versioned clean data and shared metric contract
A3         Commercial Analyst                                Demand, pricing, promotions
A4        Finance & Portfolio             Historical/projected P&L and budget comparison
A5       Inventory & Forecast      Backtest, demand forecast, cup-based inventory policy
A6             Independent QA Reconciliation, evidence, forecast and presentation checks
A7    Decision & Presentation             Recommendations, memo, charts and presentation
Design only: this notebook does not launch agents.


## แผนและคำสั่งของ agents

- [Workflow, dependencies, metrics, QA และแผน 7 วัน](../planning/workflow_plan.th.md)
- [ข้อกำหนดและ prompt ของ A0–A7](../planning/agent_blueprints.th.md)

จุดที่ต้องจัดการก่อนวิเคราะห์: ชื่อสินค้า/แถวซ้ำ/รายการนอกเมนู, ต้นทุนวันเปิดตัวที่ขาด, นิยาม budget GP, grain ของ waste และ overhead

ราคาขายรวมส่วนลดแล้ว; การเทียบช่วงโปรไม่ยืนยัน causal uplift; แผนสต็อกยังเป็นหน่วยแก้วเพราะไม่มี BOM, on-hand, lead time และ shelf life